# APS360 Dog Breed Classifier - Training on Colab

## Instructions
1. **Runtime → Change runtime type → T4 GPU** (do this first!)
2. Mount your Google Drive
3. Upload your project zip to Drive, or clone from GitHub
4. Run all cells in order

## What this notebook does
- Trains baseline ResNet-50 model (~2 hours)
- Trains primary EfficientNet-B3 model (~3 hours)
- Evaluates both models on test set
- Evaluates on new unseen data (10 points!)
- Generates all figures for final report
- Downloads results back to your machine

In [ ]:
# Cell 1: Check GPU
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU! Go to Runtime -> Change runtime type -> T4 GPU')

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Extract project from Google Drive
import os
import shutil
import glob

# ── STEP 1: Find your zip ──────────────────────────────────────────────────
# The zip was created as APS360_project.zip in your project folder.
# Upload it to the ROOT of your Google Drive (not inside any folder).
# If you put it in a subfolder, update the path below.
ZIP_PATH = '/content/drive/MyDrive/APS360_project.zip'

# ── STEP 2: Extract ────────────────────────────────────────────────────────
if not os.path.exists(ZIP_PATH):
    # Try to find it anywhere in Drive
    matches = glob.glob('/content/drive/MyDrive/**/APS360_project.zip', recursive=True)
    if matches:
        ZIP_PATH = matches[0]
        print(f'Found zip at: {ZIP_PATH}')
    else:
        raise FileNotFoundError(
            'APS360_project.zip not found in Google Drive.\n'
            'Please upload APS360_project.zip to your Google Drive root folder,\n'
            'then re-run this cell.'
        )

EXTRACT_DIR = '/content/APS360'
os.makedirs(EXTRACT_DIR, exist_ok=True)
!unzip -q "{ZIP_PATH}" -d "{EXTRACT_DIR}"

# Handle case where zip contains a subfolder
contents = os.listdir(EXTRACT_DIR)
if len(contents) == 1 and os.path.isdir(f'{EXTRACT_DIR}/{contents[0]}'):
    PROJECT_DIR = f'{EXTRACT_DIR}/{contents[0]}'
else:
    PROJECT_DIR = EXTRACT_DIR

print(f'✓ Project extracted to: {PROJECT_DIR}')
print(f'  Contents: {os.listdir(PROJECT_DIR)[:8]}')

In [ ]:
# Cell 4: Setup environment
import os
import sys

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

print(f'Working directory: {os.getcwd()}')
print(f'Key folders present: src={os.path.exists("src")}, scripts={os.path.exists("scripts")}, data={os.path.exists("data")}')

# Install dependencies
!pip install timm tqdm seaborn pyyaml pandas -q
print('✓ Dependencies installed!')

In [ ]:
# Cell 5: Verify setup
!python scripts/quick_test_new_data.py

In [ ]:
# Cell 6: Train Baseline Model (ResNet-50)
# Expected time: ~1.5-2 hours on T4 GPU
print('Starting baseline training...')
print('This will take ~1.5-2 hours on T4 GPU')
!python scripts/train_baseline.py

In [ ]:
# Cell 7: Train Primary Model (EfficientNet-B3)
# Expected time: ~2-3 hours on T4 GPU
print('Starting primary model training...')
print('This will take ~2-3 hours on T4 GPU')
!python scripts/train_primary.py

In [ ]:
# Cell 8: Evaluate on test set
!python scripts/evaluate_models.py --model_type baseline --checkpoint checkpoints/baseline/baseline_best.pth
!python scripts/evaluate_models.py --model_type primary --checkpoint checkpoints/primary/primary_best.pth

In [ ]:
# Cell 9: Evaluate on NEW DATA (10 points!)
!python scripts/evaluate_new_data.py --model_type baseline --checkpoint checkpoints/baseline/baseline_best.pth
!python scripts/evaluate_new_data.py --model_type primary --checkpoint checkpoints/primary/primary_best.pth

In [ ]:
# Cell 10: Generate all figures
!python scripts/generate_report_figures.py

In [ ]:
# Cell 11: Package results for download
import shutil
import os

# Create results package
results_dirs = ['checkpoints', 'results', 'logs']
os.makedirs('final_results_package', exist_ok=True)

for d in results_dirs:
    if os.path.exists(d):
        shutil.copytree(d, f'final_results_package/{d}', dirs_exist_ok=True)

# Zip it
shutil.make_archive('final_results', 'zip', 'final_results_package')
print('Results packaged: final_results.zip')

# Copy to Drive for easy download
shutil.copy('final_results.zip', '/content/drive/MyDrive/APS360_final_results.zip')
print('Copied to Google Drive: APS360_final_results.zip')

In [ ]:
# Cell 12: Show summary of results
import json
import os

print('='*60)
print('TRAINING COMPLETE - RESULTS SUMMARY')
print('='*60)

# Load and display results
for model_type in ['baseline', 'primary']:
    history_file = f'checkpoints/{model_type}/{model_type}_history.json'
    if os.path.exists(history_file):
        with open(history_file) as f:
            history = json.load(f)
        best_val_acc = max(history['val_acc'])
        best_epoch = history['val_acc'].index(best_val_acc) + 1
        print(f'\n{model_type.upper()} Model:')
        print(f'  Best Val Accuracy: {best_val_acc:.2f}% (epoch {best_epoch})')
        if 'val_top5_acc' in history:
            best_top5 = max(history['val_top5_acc'])
            print(f'  Best Top-5 Accuracy: {best_top5:.2f}%')

print('\n' + '='*60)
print('Download APS360_final_results.zip from your Google Drive!')
print('='*60)